# Fine-tuning LLaMA 3.2 3B - Medical Assistant
**Plataforma: Kaggle (GPU T4 gratuita, 30h/semana)**

Este notebook realiza o fine-tuning do modelo `meta-llama/Llama-3.2-3B-Instruct`
com a tecnica QLoRA (Quantized Low-Rank Adaptation) usando o dataset MedQuAD.

O que e QLoRA:
- Em vez de treinar todos os 3 bilhoes de parametros do modelo (impossivel com pouca VRAM),
  o QLoRA congela o modelo e adiciona pequenas matrizes de adaptacao (adaptadores LoRA)
  em camadas especificas. Apenas esses adaptadores sao treinados.
- O modelo e carregado em 4-bit (quantizado), reduzindo o uso de memoria de ~14GB para ~4GB.
- Resultado: apenas 0.17% dos parametros sao treinados, mas o modelo aprende o dominio medico.

Ao final, os adaptadores LoRA ficam salvos em /kaggle/working/llama-medical/.
Esses arquivos devem ser baixados e colocados em outputs/llama-medical/ no projeto local.

Pre-requisitos para rodar este notebook:
- Conta no Kaggle com telefone verificado (para acessar GPU)
- Acesso aprovado ao modelo no HuggingFace: huggingface.co/meta-llama/Llama-3.2-3B-Instruct
- Arquivos train.jsonl e val.jsonl gerados pelo dataset_builder.py


## Celula 1 - Verificar GPU disponivel

In [ ]:
import torch

print("GPU disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))
    print("VRAM total:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("GPU nao encontrada.")
    print("No Kaggle: clique em Session Options > Accelerator > GPU T4 x2")


## Celula 2 - Definir pasta de saida
> Os adaptadores LoRA serao salvos em /kaggle/working/llama-medical/

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/llama-medical"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Pasta de saida: {OUTPUT_DIR}")


## Celula 3 - Instalar dependencias

In [ ]:
!pip install -q torch transformers peft bitsandbytes accelerate trl datasets
print("Dependencias instaladas.")


## Celula 4 - Login no HuggingFace
O modelo LLaMA 3.2 e um modelo restrito (gated). Para baixa-lo, voce precisa:
1. Ter solicitado acesso em: huggingface.co/meta-llama/Llama-3.2-3B-Instruct
2. Gerar um token READ em: huggingface.co/settings/tokens
3. Colar o token abaixo


In [ ]:
from huggingface_hub import login

HF_TOKEN = ""  # Cole seu token READ aqui
login(token=HF_TOKEN)
print("Login HuggingFace realizado.")


## Celula 5 - Upload do dataset
Faca upload dos arquivos train.jsonl e val.jsonl gerados localmente pelo dataset_builder.py.

No Kaggle, clique em "+ Add Input" no painel direito, depois em "Upload",
e selecione os dois arquivos. Apos o upload, rode a celula abaixo para confirmar.


In [ ]:
import os

# Substitua pelo caminho correto apos o upload
# Exemplo: /kaggle/input/medquad-processed/train.jsonl
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))


## Celula 6 - Carregar modelo com QLoRA (4-bit)
O modelo e carregado com quantizacao 4-bit (BitsAndBytes), reduzindo o uso de VRAM
de ~14GB para ~4GB. Isso permite rodar na T4 gratuita do Kaggle.

O modelo base e o Llama-3.2-3B-Instruct. Usamos bfloat16 como tipo de dado
porque o LLaMA 3.2 foi treinado originalmente nesse formato e a T4 o suporta.


In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

# Configuracao de quantizacao 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4: melhor qualidade para quantizacao de pesos
    bnb_4bit_compute_dtype=torch.bfloat16,  # Tipo usado nos calculos internos
    bnb_4bit_use_double_quant=False,
)

print(f"Carregando {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token  # LLaMA nao tem pad token, usa eos
tokenizer.padding_side = "right"              # Padding a direita para causal LM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
print("Modelo carregado.")


## Celula 7 - Configurar adaptadores LoRA
O LoRA adiciona matrizes de baixo rank (r=16) nas camadas de atencao do modelo.
Apenas essas matrizes serao treinadas, enquanto o resto do modelo fica congelado.

Parametros:
- r=16: rank das matrizes LoRA (controla capacidade vs. tamanho dos adaptadores)
- lora_alpha=32: escala do aprendizado LoRA (geralmente 2x o rank)
- target_modules: camadas onde os adaptadores sao inseridos (todas as projecoes de atencao)
- lora_dropout=0.05: dropout para evitar overfitting


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

torch.cuda.empty_cache()
model.config.use_cache = False
model.enable_input_require_grads()

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Resultado esperado: trainable params ~13M de 3B total (~0.17%)


## Celula 8 - Fine-tuning
O treinamento usa o SFTTrainer (Supervised Fine-Tuning Trainer) da biblioteca TRL.

Parametros importantes:
- max_steps=100: limita o treino para demonstracao. Para treino completo, remova essa linha.
- gradient_checkpointing=True: economiza VRAM trocando por tempo de processamento
- bf16=True: usa bfloat16 (compativel com LLaMA 3.2, evita erros com fp16)
- gradient_accumulation_steps=4: acumula gradientes de 4 steps antes de atualizar
  (simula batch size maior sem usar mais VRAM)
- output_dir=OUTPUT_DIR: salva checkpoints diretamente na pasta de saida do Kaggle

Nota: o output_dir aponta para /kaggle/working/llama-medical/ para que os
checkpoints sejam salvos junto com o modelo final.


In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
import torch

torch.cuda.empty_cache()

# Substitua pelos caminhos corretos dos arquivos que voce fez upload
dataset = load_dataset("json", data_files={
    "train":      "/kaggle/input/SEU_DATASET/train.jsonl",  # ajuste o caminho
    "validation": "/kaggle/input/SEU_DATASET/val.jsonl",    # ajuste o caminho
})
print(f"Treino: {len(dataset['train'])} | Validacao: {len(dataset['validation'])}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    bf16=True,
    optim="paged_adamw_32bit",
    report_to="none",
    load_best_model_at_end=True,
    gradient_checkpointing=True,
    max_steps=100,  # remova para treino completo (demora ~2h)
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

print("Iniciando fine-tuning...")
trainer.train()
print("Fine-tuning concluido.")


## Celula 9 - Salvar modelo

In [ ]:
import shutil

print(f"Salvando adaptadores LoRA em: {OUTPUT_DIR}")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

files = os.listdir(OUTPUT_DIR)
print(f"Arquivos salvos ({len(files)}):")
for f in files:
    print(f"  {f}")

# Compacta a pasta para facilitar o download
zip_path = "/kaggle/working/llama-medical"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"\nArquivo ZIP criado: {zip_path}.zip")
print("Clique em 'Output' no painel direito para baixar.")


## Celula 10 - Teste rapido do modelo treinado
Verifica se o modelo responde adequadamente a perguntas clinicas basicas.


In [ ]:
from transformers import pipeline as hf_pipeline

pipe = hf_pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=False,
)

perguntas = [
    "What is the recommended treatment for type 2 diabetes?",
    "What are the symptoms of pneumonia?",
    "How is hypertension diagnosed?",
]

for q in perguntas:
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{q}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    result = pipe(prompt)[0]["generated_text"]
    answer = result.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    answer = answer.replace("<|eot_id|>", "").strip()
    print(f"Pergunta: {q}")
    print(f"Resposta: {answer[:300]}")
    print()
